# Notebook 8 — Geospatial Visualization

Interactive world maps and district-level heatmaps showing malnutrition risk levels.
These visualizations enable policymakers to identify high-risk regions and prioritize interventions.

In [ ]:
# from google.colab import drive  # removed for local run
# drive.mount('/content/drive')  # removed for local run

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

PROCESSED = '../data/processed'
OUTPUTS = '../outputs/plots'

# Load data
data = pd.read_csv(f'{PROCESSED}/main_clustered.csv')
india_district = pd.read_csv(f'{PROCESSED}/india_district.csv')

print(f"Global data: {data.shape}")
print(f"India district data: {india_district.shape}")
print(f"\nRisk levels in data: {data['risk_label'].unique()}")

## Phase 1: World Map - Risk Levels by Country

Create an interactive choropleth map showing malnutrition risk levels across 150+ countries.

In [ ]:
# Map risk labels to numeric values for visualization
risk_mapping = {
    'Low Risk': 1,
    'Moderate Risk': 2,
    'High Risk': 3,
    'Severe Risk': 4,
    'Critical Risk': 5
}
risk_colors = {
    'Low Risk': '#2ecc71',
    'Moderate Risk': '#f39c12',
    'High Risk': '#e74c3c',
    'Severe Risk': '#c0392b',
    'Critical Risk': '#8b0000'
}

data['risk_numeric'] = data['risk_label'].map(risk_mapping)

# Get latest year data and aggregate by country
latest_year = data['year'].max()
country_risk = data[data['year'] == latest_year][['Country', 'risk_label', 'risk_numeric', 
                                                      'stunting', 'wasting', 'underweight']].drop_duplicates()

# Create world choropleth map using Plotly
fig = go.Figure(data=go.Choropleth(
    locations = country_risk['Country'],
    z = country_risk['risk_numeric'],
    locationmode = 'country names',
    text = country_risk['risk_label'],
    colorscale = [[0, '#2ecc71'], [0.25, '#f39c12'], [0.5, '#e74c3c'], [0.75, '#c0392b'], [1, '#8b0000']],
    zmin = 1,
    zmax = 5,
    colorbar = dict(title="Risk Level", tickvals=[1,2,3,4,5], 
                    ticktext=['Low', 'Moderate', 'High', 'Severe', 'Critical']),
    hovertemplate = '<b>%{text}</b><br>Risk Level: %{z}<extra></extra>',
    name=''
))

fig.update_layout(
    title_text=f'Global Malnutrition Risk Assessment ({latest_year})',
    geo=dict(projection_type='natural earth', bgcolor='rgba(200, 200, 200, 0.3)'),
    height=600,
    width=1200
)

fig.write_html(f'{OUTPUTS}/world_risk_map.html')
fig.show()
print("✓ World risk map saved → world_risk_map.html")

## Phase 2: Region-wise Comparison

Bar chart showing average malnutrition indicators by region.

In [ ]:
# Aggregate by region
region_stats = data.groupby('region')[['stunting', 'wasting', 'underweight', 'risk_numeric']].mean()
region_stats = region_stats.sort_values('risk_numeric', ascending=False)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Stunting by region
axes[0, 0].barh(region_stats.index, region_stats['stunting'], color='steelblue')
axes[0, 0].set_xlabel('Average Stunting %')
axes[0, 0].set_title('Stunting by Region')
axes[0, 0].grid(axis='x', alpha=0.3)

# Wasting by region
axes[0, 1].barh(region_stats.index, region_stats['wasting'], color='coral')
axes[0, 1].set_xlabel('Average Wasting %')
axes[0, 1].set_title('Wasting by Region')
axes[0, 1].grid(axis='x', alpha=0.3)

# Underweight by region
axes[1, 0].barh(region_stats.index, region_stats['underweight'], color='lightcoral')
axes[1, 0].set_xlabel('Average Underweight %')
axes[1, 0].set_title('Underweight by Region')
axes[1, 0].grid(axis='x', alpha=0.3)

# Risk level by region
risk_colors_list = [risk_colors[risk] for risk in data.groupby('region')['risk_label'].apply(lambda x: x.mode()[0] if len(x.mode()) > 0 else 'Low Risk')]
axes[1, 1].barh(region_stats.index, region_stats['risk_numeric'], color=risk_colors_list)
axes[1, 1].set_xlabel('Average Risk Level')
axes[1, 1].set_title('Risk Level by Region')
axes[1, 1].set_xticks([1, 2, 3, 4, 5])
axes[1, 1].set_xticklabels(['Low', 'Mod', 'High', 'Sev', 'Crit'])
axes[1, 1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUTPUTS}/region_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Region analysis saved → region_analysis.png")

## Phase 3: India District-Level Heatmap

District-level visualization showing rainfall, crop production, and food security indicators across Indian states.

In [ ]:
# Aggregate india data by district
india_agg = india_district.groupby(['state', 'district'])[['annual_rainfall', 'monsoon_rainfall', 
                                                               'crop_production']].mean().reset_index()

# Normalize for heatmap
india_agg['crop_score'] = (india_agg['crop_production'] - india_agg['crop_production'].min()) / \
                           (india_agg['crop_production'].max() - india_agg['crop_production'].min())
india_agg['rainfall_score'] = (india_agg['annual_rainfall'] - india_agg['annual_rainfall'].min()) / \
                               (india_agg['annual_rainfall'].max() - india_agg['annual_rainfall'].min())

# Create state-level summary for heatmap
state_summary = india_agg.groupby('state')[['crop_score', 'rainfall_score']].mean()
state_summary = state_summary.sort_values('crop_score', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 8))

# Crop production by state
sns.heatmap(pd.DataFrame(state_summary['crop_score']).T, annot=True, fmt='.2f', 
            cmap='YlGn', ax=axes[0], cbar_kws={'label': 'Crop Production Score'})
axes[0].set_title('Crop Production by State')
axes[0].set_xlabel('State')
plt.setp(axes[0].get_xticklabels(), rotation=45, ha='right')

# Rainfall by state
sns.heatmap(pd.DataFrame(state_summary['rainfall_score']).T, annot=True, fmt='.2f', 
            cmap='Blues', ax=axes[1], cbar_kws={'label': 'Rainfall Score'})
axes[1].set_title('Rainfall Distribution by State')
axes[1].set_xlabel('State')
plt.setp(axes[1].get_xticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.savefig(f'{OUTPUTS}/india_district_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ India district heatmap saved → india_district_heatmap.png")

## Phase 4: Risk Distribution by Continent

Pie charts showing risk level distribution across continents.

In [ ]:
# Map countries to continents (simplified mapping)
continent_mapping = {
    'India': 'Asia', 'China': 'Asia', 'Indonesia': 'Asia', 'Bangladesh': 'Asia', 'Pakistan': 'Asia',
    'Nigeria': 'Africa', 'Ethiopia': 'Africa', 'Congo': 'Africa', 'Tanzania': 'Africa', 'Kenya': 'Africa',
    'Brazil': 'South America', 'Peru': 'South America', 'Colombia': 'South America',
    'USA': 'North America', 'Mexico': 'North America',
    'Germany': 'Europe', 'France': 'Europe', 'UK': 'Europe',
    'Australia': 'Oceania'
}

data['continent'] = data['Country'].map(continent_mapping).fillna('Other')

# Count risk levels by continent
continent_risk = pd.crosstab(data['continent'], data['risk_label'])

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

continents = data['continent'].unique()
colors = ['#2ecc71', '#f39c12', '#e74c3c', '#c0392b', '#8b0000']

for idx, continent in enumerate(continents[:6]):
    if continent in continent_risk.index:
        risk_data = continent_risk.loc[continent]
        axes[idx].pie(risk_data, labels=risk_data.index, autopct='%1.1f%%', colors=colors, startangle=90)
axes[idx].set_title(f'{continent}')

plt.suptitle('Risk Level Distribution by Continent', fontsize=16, y=1.00)
plt.tight_layout()
plt.savefig(f'{OUTPUTS}/continental_risk_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Continental risk distribution saved → continental_risk_distribution.png")

## Phase 5: Top 10 Critical Risk Countries

Highlight countries with Critical or Severe risk levels requiring urgent intervention.

In [ ]:
# Get top critical risk countries
critical_countries = data[data['risk_label'].isin(['Critical Risk', 'Severe Risk'])]
critical_summary = critical_countries.groupby('Country')[['stunting', 'wasting', 'underweight']].mean()
critical_summary['country_count'] = critical_countries.groupby('Country').size()
critical_summary = critical_summary.sort_values('stunting', ascending=False).head(10)

fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(critical_summary))
width = 0.25

ax.bar(x - width, critical_summary['stunting'], width, label='Stunting %', color='#e74c3c')
ax.bar(x, critical_summary['wasting'], width, label='Wasting %', color='#c0392b')
ax.bar(x + width, critical_summary['underweight'], width, label='Underweight %', color='#8b0000')

ax.set_xlabel('Country')
ax.set_ylabel('Percentage')
ax.set_title('Top 10 Critical Risk Countries - Malnutrition Indicators')
ax.set_xticks(x)
ax.set_xticklabels(critical_summary.index, rotation=45, ha='right')
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUTPUTS}/critical_risk_countries.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Critical risk countries visualization saved → critical_risk_countries.png")
print(f"\nTop 10 Critical/Severe Risk Countries:")
print(critical_summary)

## Phase 6: Interactive Plotly Dashboard

Combining multiple visualizations into an interactive dashboard.

In [ ]:
from plotly.subplots import make_subplots

# Create subplots dashboard
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Risk Distribution', 'Top Critical Countries', 'Region Analysis', 'Temporal Trends'),
    specs=[[{'type': 'pie'}, {'type': 'bar'}],
           [{'type': 'bar'}, {'type': 'scatter'}]]
)

# Risk distribution pie
risk_counts = data['risk_label'].value_counts()
fig.add_trace(
    go.Pie(labels=risk_counts.index, values=risk_counts.values, name='Risk Distribution'),
    row=1, col=1
)

# Top critical countries bar
top_critical = critical_summary.head(5).reset_index()
fig.add_trace(
    go.Bar(x=top_critical['Country'], y=top_critical['stunting'], name='Stunting %'),
    row=1, col=2
)

# Region analysis bar
region_risk = data.groupby('region')['risk_numeric'].mean().sort_values(ascending=False)
fig.add_trace(
    go.Bar(x=region_risk.index, y=region_risk.values, name='Avg Risk'),
    row=2, col=1
)

# Temporal trends
yearly_average = data.groupby('year')['risk_numeric'].mean()
fig.add_trace(
    go.Scatter(x=yearly_average.index, y=yearly_average.values, mode='lines+markers', name='Avg Risk'),
    row=2, col=2
)

fig.update_layout(height=900, showlegend=False, title_text="Global Malnutrition Risk Dashboard")
fig.write_html(f'{OUTPUTS}/interactive_dashboard.html')
fig.show()
print("✓ Interactive dashboard saved → interactive_dashboard.html")

## Notebook 8 — Complete

Geospatial visualizations provide critical insights for policy decisions:
- **World choropleth map**: At-a-glance view of global malnutrition risk
- **Regional analysis**: Identify high-burden regions (Sub-Saharan Africa, South Asia)
- **District-level heatmaps**: Granular insights for India's 641 districts
- **Critical country spotlight**: Prioritize international aid and intervention

**Outputs Saved:**
- outputs/plots/world_risk_map.html → Interactive world map
- outputs/plots/region_analysis.png → Regional breakdown
- outputs/plots/india_district_heatmap.png → District-level analysis
- outputs/plots/continental_risk_distribution.png → Continental comparison
- outputs/plots/critical_risk_countries.png → Top 10 critical countries
- outputs/plots/interactive_dashboard.html → Combined dashboard